# 01 — Exploratory Data Analysis

SE4050 Deep Learning. Feeds **Section 3 (Dataset Description and EDA)** of the report.

**Run these two commands first**, from the project root:

```bash
python get_data.py --out data/ --per-class 15000
python src/data.py --input data/complaints_subset.csv --out data/
```

This notebook only reads the resulting splits. It downloads nothing and
changes nothing, so it is safe to re-run.

Every figure is saved to `results/figures/`. The marks are for the
**interpretation**, not the plot — each cell prints the sentence you need.

In [ ]:
import os, sys

# run from the project root, not from notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_splits

os.makedirs('results/figures', exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
BLUE = '#4269d0'

train, val, test, meta = load_splits('data')
print(f"train {len(train):,} | val {len(val):,} | test {len(test):,}")
print(f"classes: {meta['n_classes']}   seed: {meta['seed']}")

## 1. Leakage check

Both numbers must be **0**. This is the evidence for the preprocessing marks —
screenshot this output for the report.

In [ ]:
print('train ∩ test :', len(set(train.text) & set(test.text)))
print('train ∩ val  :', len(set(train.text) & set(val.text)))
print('val   ∩ test :', len(set(val.text) & set(test.text)))

# the splits must also hold the same class proportions
props = pd.DataFrame({
    'train': train.label.value_counts(normalize=True),
    'val':   val.label.value_counts(normalize=True),
    'test':  test.label.value_counts(normalize=True),
}).round(4)
props

## 2. Class distribution

How imbalanced is the target?

In [ ]:
counts = train['label'].value_counts()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh([c[:42] for c in counts.index][::-1], counts.values[::-1], color=BLUE)
ax.set_xlabel('complaints in the training set')
ax.set_title('Class distribution')
fig.tight_layout()
fig.savefig('results/figures/class_distribution.png', bbox_inches='tight')

ratio = counts.max() / counts.min()
print(counts.to_string())
print(f'\nimbalance ratio = {ratio:.1f} : 1')
print(f'a model predicting only "{counts.index[0][:40]}" would score '
      f'{counts.max()/counts.sum():.1%} accuracy and near-zero macro F1')
print('-> report macro F1 as the headline metric, and use class-weighted loss')

## 3. Narrative length

This decides MAX_LEN for every model.

In [ ]:
lengths = train['text'].str.split().str.len()
p50, p90, p95 = (int(np.percentile(lengths, p)) for p in (50, 90, 95))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lengths, bins=60, range=(0, min(800, lengths.max())), color=BLUE)
for v, c, lab in [(p50, '#efb118', f'p50 = {p50}'),
                  (p90, '#ff725c', f'p90 = {p90}'),
                  (p95, '#6cc5b0', f'p95 = {p95}')]:
    ax.axvline(v, color=c, ls='--', lw=1.6, label=lab)
ax.set_xlabel('words per complaint'); ax.set_ylabel('complaints')
ax.set_title('Narrative length'); ax.legend()
fig.tight_layout()
fig.savefig('results/figures/length_distribution.png', bbox_inches='tight')

print(f'median {p50} | p90 {p90} | p95 {p95} | max {lengths.max()}')
print(f'-> MAX_LEN = {p90}: covers 90% of complaints in full. Going to '
      f'{lengths.max()} would multiply compute for the last 10%.')

## 4. Length by category

In [ ]:
order = counts.index.tolist()
data = [train.loc[train.label == c, 'text'].str.split().str.len() for c in order]

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(data, vert=False, labels=[c[:34] for c in order], showfliers=False)
ax.set_xlabel('words per complaint')
ax.set_title('Complaint length by product')
fig.tight_layout()
fig.savefig('results/figures/length_by_class.png', bbox_inches='tight')

med = train.assign(n=lengths).groupby('label')['n'].median().sort_values()
print('median length by class:'); print(med.to_string())
print('\n-> categories with much longer narratives favour models that handle '
      'long context (BiLSTM, Transformer) over TextCNN')

## 5. Which words mark each category

Use this later to explain the confusion matrix: categories that share their
top words are the ones your models will mix up.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=20000, stop_words='english', min_df=5)
X = tfidf.fit_transform(train['text'])
vocab = np.array(tfidf.get_feature_names_out())

for c in order:
    mask = (train['label'] == c).values
    score = np.asarray(X[mask].mean(0)).ravel() - np.asarray(X[~mask].mean(0)).ravel()
    top = ', '.join(vocab[score.argsort()[-8:][::-1]])
    print(f'{c[:44]:46s} {top}')

## 6. Vocabulary and data quality

In [ ]:
from collections import Counter

words = Counter()
train['text'].str.split().map(words.update)

total_unique = len(words)
for size in (10000, 20000, 30000, 50000):
    covered = sum(f for _, f in words.most_common(size))
    print(f'top {size:>6,} words cover {covered/sum(words.values()):.2%} of all tokens')

print(f'\nunique words: {total_unique:,}')
print(f'appearing once only: {sum(1 for f in words.values() if f == 1):,}')
print('-> 30,000 is enough for the vocabulary; rarer words become <unk>')

In [ ]:
# a few real complaints — read them, they explain your errors later
for c in order[:4]:
    row = train[train.label == c].iloc[0]
    print(f'[{c}]\n{row.text[:300]}...\n')

## Fill this in for the report

| Item | Value |
| --- | --- |
| Snapshot download date | |
| Complaints with a narrative | |
| After cleaning and dedupe | |
| Classes kept | |
| Imbalance ratio | |
| MAX_LEN (p90) | |
| Train / val / test | |

**Data-quality issues to name:** personal details masked as `XXXX`; duplicate
submissions removed before splitting; product categories renamed by the CFPB
over time and merged here; heavy class imbalance; narratives ranging from one
line to several pages.